## Open notebook in:
| Colab                                 
:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH06/ch06_Qwen2_Audio_different_audio_tasks.ipynb)                                             

# About this Notebook

This notebook showcases how to interact with **Qwen2-Audio-7B-Instruct**, a large language model capable of understanding and generating responses based on both **audio** and **text** inputs. The model allows for multimodal chat-style interactions with audio clips, making it well-suited for tasks such as **speaker analysis**, **sound event detection**, and **speech transcription**.

### Steps Included:

1. **Model and Processor Setup**:
   The notebook loads the `Qwen2-Audio-7B-Instruct` model and its corresponding processor from the Hugging Face Hub. The model is loaded with automatic device mapping, using the GPU if available.

2. **Prompt Formatting**:
   Chat-style prompts are constructed using the processor's `apply_chat_template` function. These prompts can contain both audio clips (remote URLs or local files) and textual queries.

3. **Audio Preprocessing**:
   Audio clips are downloaded or loaded from local storage using `librosa`, then resampled to match the model’s expected sampling rate.

4. **Batch Input Construction**:
   Both the textual prompts and the corresponding audio waveforms are tokenized and packed into a batch format using the processor. The inputs are padded and transferred to the appropriate device.

5. **Inference and Response Generation**:
   The model generates a textual response to each multimodal prompt using the `generate` method. The response is then decoded into human-readable text.

6. **Use Cases Demonstrated**:

   * **Gender and age prediction** from voice
   * **Sound recognition** (e.g., detecting glass breaking or gunshots)
   * **Speech transcription** from audio files

This notebook illustrates how modern multimodal language models can move beyond text and process **real-world audio inputs** with flexible, instruction-following behavior, bridging the gap between natural language understanding and audio signal processing.


# Installs

In [ ]:
!pip install -U transformers gdown bitsandbytes -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.9 MB/s eta 0:00:00


# Imports

In [ ]:
from io import BytesIO
from urllib.request import urlopen
import os
import librosa
import torch
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

# Load model and processor

In [ ]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-Audio-7B-Instruct")
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-Audio-7B-Instruct", device_map="auto", load_in_4bit=True,
)
# from transformers import BitsAndBytesConfig
# processor = AutoProcessor.from_pretrained("alicekyting/Qwen2-Audio-7B-Instruct-4bit")
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16
# )
# model = Qwen2AudioForConditionalGeneration.from_pretrained(
#     "alicekyting/Qwen2-Audio-7B-Instruct-4bit",
#     device_map="auto",
#     quantization_config=bnb_config
# )

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/638k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/212k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/876 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/248 [00:00<?, ?B/s]

# Load and prepare audio data

In [ ]:

# Download the file from Google Drive using its file ID
file_id = "1IymJaN-zOk7YgZCWPsTmzK2lp45mgfjL"
destination = "gunshots.wav"
!gdown --id {file_id} -O {destination}

# Check if the file was downloaded
import os
if os.path.exists(destination):
    print(f"File downloaded successfully: {destination}")
else:
    print("Download failed.")


usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--proxy PROXY] [--speed SPEED]
             [--no-cookies] [--no-check-certificate] [--continue] [--folder]
             [--json] [--format FORMAT] [--user-agent USER_AGENT]
             url_or_id
gdown: error: unrecognized arguments: --id
File downloaded successfully: gunshots.wav


In [ ]:
conversation1 = [
    {"role": "user", "content": [
        {"type": "audio", "audio_url": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/glass-breaking-151256.mp3"},
        {"type": "text", "text": "What's that sound?"},
    ]},
]

conversation2 = [
    {"role": "user", "content": [
        {"type": "audio", "audio_url": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/1272-128104-0000.flac"},
        {"type": "text", "text": "What does the person say?"},
    ]},
]

conversation3 = [
    {"role": "user", "content": [
        {"type": "audio", "audio_url": "/content/gunshots.wav"},
        {"type": "text", "text": "What is that sound, and what does the person say?"},
    ]},
]
conversation4 = [
    {"role": "user", "content": [
        {"type": "audio", "audio_url": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/guess_age_gender.wav"},
        {"type": "text", "text": "Can you guess the speaker's gender and age?"}
    ]},
]

conversations = [conversation1, conversation2, conversation3,conversation4]


# Process and predict

In [ ]:

# Prepare inputs
text = [processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False) for conv in conversations]

sr = processor.feature_extractor.sampling_rate
audios = []

for conversation in conversations:
    for message in conversation:
        if isinstance(message["content"], list):
            for ele in message["content"]:
                if ele["type"] == "audio":
                    audio_path = ele['audio_url']
                    if audio_path.startswith("http"):
                        # Remote URL
                        audio, _ = librosa.load(BytesIO(urlopen(audio_path).read()), sr=sr)
                    else:
                        # Load local file
                        if not os.path.exists(audio_path):
                            raise FileNotFoundError(f"Local audio file not found: {audio_path}")
                        audio, _ = librosa.load(audio_path, sr=sr)
                    audios.append(audio)

# Batch processing
inputs = processor(text=text, audio=audios, return_tensors="pt", padding=True)

# Move all tensor inputs to the correct device
device = model.device
inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

# Generate response
# model.eval()
# with torch.inference_mode():
generate_ids = model.generate(**inputs, max_new_tokens=512)
generate_ids = generate_ids[:, inputs["input_ids"].size(1):]

# Decode
responses = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

# Print outputs
for i, r in enumerate(responses):
    print(f"Response {i+1}: {r}")


[transformers] It is strongly recommended to pass the `sampling_rate` argument to `WhisperFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.


Response 1: Shattered glass.
Response 2: The original content of this audio is: 'Mister Quiller is the apostle of the middle classes, and we are glad to welcome his gospel.'
Response 3: In the audio, there is the sound of gunfire and artillery fire at intervals (0.52-1.37), (3.64-4.89), and (6.47-10.00). A male voice speaks in English saying 'Can you guess where I am right now?' with a neutral mood.
Response 4: The speaker is female and in her twenties.
